In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. CSV'i oku
df = pd.read_csv("C:\\Users\\catsu\\PycharmProjects\\scrappyhotel\\final_processed_reviews_deepl_use.csv", encoding="utf-8-sig")

# 2. Gerekli sütunları temizle
df = df.dropna(subset=["hotel_name", "processed_final_review"])

# 3. Otel bazında genel puan (hotel_overall_rating) al ve tekrar edenleri kaldırıver
hotels = (
    df[["hotel_name", "hotel_overall_rating"]]
    .drop_duplicates(subset="hotel_name")
    .reset_index(drop=True)
)

# 4. Her otelin tüm temizlenmiş yorumlarını birleştir
agg = (
    df.groupby("hotel_name")["processed_final_review"]
      .agg(lambda texts: " ".join(texts))
      .reset_index()
      .merge(hotels, on="hotel_name", how="left")
)

# 5. TF‑IDF vektörizasyonu
tfidf = TfidfVectorizer(min_df=2, max_df=0.8, ngram_range=(1,2))
X = tfidf.fit_transform(agg["processed_final_review"])

# 6. Cosine‑similarity matrisi
cos_sim = cosine_similarity(X, X)

# 7. Hızlı erişim için isim→indeks haritası
name_to_idx = pd.Series(data=agg.index, index=agg["hotel_name"])

def recommend_similar(hotel_name, top_n=5):
    idx = name_to_idx[hotel_name]
    sims = list(enumerate(cos_sim[idx]))
    sims = sorted(sims, key=lambda x: x[1], reverse=True)[1:top_n+1]
    return [
        {
          "hotel_name": agg["hotel_name"].iloc[i],
          "overall_rating": agg["hotel_overall_rating"].iloc[i],
          "similarity": float(score)
        }
        for i, score in sims
    ]

# Örnek:
print("AnkAkyaka Apart Hotel için en yakın 5:")
for rec in recommend_similar("AnkAkyaka Apart Hotel", 5):
    print(rec)


AnkAkyaka Apart Hotel için en yakın 5:
{'hotel_name': 'Koparan Apart', 'overall_rating': 4.3, 'similarity': 0.20210702556368945}
{'hotel_name': 'Santo Akyaka Home', 'overall_rating': 4.1, 'similarity': 0.1995053358401537}
{'hotel_name': 'Summer Otel', 'overall_rating': 4.0, 'similarity': 0.17883145550545976}
{'hotel_name': 'Panamare Apart & Hotel Akyaka', 'overall_rating': 4.6, 'similarity': 0.16825582981718132}
{'hotel_name': 'İnci Apart', 'overall_rating': 4.4, 'similarity': 0.1681994612110316}
